# Q2 - Hardware-aware Evolutionary Neural Architecture Search on CIFAR-10 (Google Colab runner)

This notebook:
1. installs the libraries,
2. lets you **upload the .py files** of this folder,
3. downloads CIFAR-10,
4. runs the search, the final training, ONNX export, inference, benchmark and plots,
5. zips the `output` folder so you can download it.

**Before you start:** Runtime -> Change runtime type -> **T4 GPU**.
(The CPU latency and size of every candidate are measured on the Colab CPU, so the constraint
limits in `config.py` refer to that CPU.)

## Step 1 - Install libraries

In [ ]:
# onnx / onnxruntime: export + CPU inference. psutil: peak-memory measurement.
!pip install -q onnx onnxruntime psutil

## Step 2 - Upload the python files
Run the cell and select **all .py files** from the Q2 folder
(config.py, search_space.py, model_builder.py, data.py, hardware.py, trainer.py, evolution.py,
train_best.py, export_onnx.py, onnx_inference.py, benchmark.py, plot.py).

In [ ]:
import os
from google.colab import files

# Work inside a dedicated folder so all outputs end up in /content/q2/output
WORK_DIR = "/content/q2"
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

# Opens a file picker; the selected files are saved in the current folder.
uploaded = files.upload()

required = ["config.py", "search_space.py", "model_builder.py", "data.py", "hardware.py",
            "trainer.py", "evolution.py", "train_best.py", "export_onnx.py",
            "onnx_inference.py", "benchmark.py", "plot.py"]
missing = [f for f in required if not os.path.exists(f)]
print("Missing files:", missing if missing else "none - all files uploaded")

## Step 3 - Load the CIFAR-10 dataset

In [ ]:
import torchvision

# Downloads and extracts CIFAR-10 into /content/q2/data/cifar-10-batches-py
for is_train in (True, False):
    torchvision.datasets.CIFAR10(root="data", train=is_train, download=True)
print("Dataset ready.")

## Step 4 - Run the evolutionary architecture search
Measures CPU latency / size / peak memory of every candidate, rejects the ones that violate the
constraints, trains the rest briefly and evolves the population.
Writes `output/search_history.csv`, `output/search_history.json` and `output/best_architecture.json`.

In [ ]:
!python evolution.py

## Step 5 - Train the best architecture from scratch

In [ ]:
!python train_best.py

## Step 6 - Export the best model to ONNX

In [ ]:
!python export_onnx.py

## Step 7 - ONNX Runtime CPU inference (test-set accuracy)

In [ ]:
!python onnx_inference.py

## Step 8 - Benchmark (writes `output/benchmark.csv`)

In [ ]:
!python benchmark.py
import pandas as pd
pd.read_csv("output/benchmark.csv")

## Step 9 - Plots (one per recorded metric)

In [ ]:
!python plot.py
from IPython.display import Image, display
for name in ["val_accuracy", "latency_ms", "model_size_mb", "peak_memory_mb", "train_loss", "test_accuracy"]:
    path = f"output/plots/{name}.png"
    if os.path.exists(path):
        display(Image(path))

## Step 10 - Download everything in `output/`

In [ ]:
# Zips the whole output folder and downloads it to your computer.
!zip -r -q q2_output.zip output
files.download("q2_output.zip")